# 🔬 PATH A1: Merge QLoRA Adapter → Upload Merged Gemma-4 E4B to HF Hub

- **LoRA Adapter:** `hung2903/gemma-4-E4B-unsloth-vaccine-xai`
- **Base Model:** `unsloth/gemma-4-E4B-it`
- **Output Repo:** `hung2903/gemma-4-E4B-vaccine-xai-merged`

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q huggingface_hub

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"GPU count: {torch.cuda.device_count()}")

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)
print("✅ Logged in to HuggingFace Hub")

In [ ]:
LORA_ADAPTER = "hung2903/gemma-4-E4B-unsloth-vaccine-xai"
UNSLOTH_BASE = "unsloth/gemma-4-E4B-it"
MERGED_REPO = "hung2903/gemma-4-E4B-vaccine-xai-merged"
MAX_SEQ_LENGTH = 2048
SAVE_DIR = "/kaggle/working/merged_16bit"

In [ ]:
# ✅ FIX: KHÔNG gọi FastModel.for_inference() ở đây.
# for_inference() patch internal state của model và không tương thích
# với save_pretrained_merged() — sẽ gây lỗi hoặc cho ra merged model bị corrupt.
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print(f"Loading adapter: {LORA_ADAPTER}")
print(f"Base: {UNSLOTH_BASE}")
print("⏳ This will download ~50MB adapter + ~5GB base...")

model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_ADAPTER,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
)
# ❌ KHÔNG gọi: FastModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

print(f"\n✅ Loaded successfully")
print(f"Model class: {type(model).__name__}")
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# ✅ FIX: Dùng model.eval() thủ công cho inference test thay vì for_inference().
# Sau khi test xong, model vẫn ở trạng thái tương thích với merge.
test_prompt = """Bạn là Trí tuệ Nhân tạo có khả năng giải thích (Explainable AI) trong lĩnh vực Y tế Công cộng. Hãy phân tích văn bản sau đây về chủ đề vắc-xin, đưa ra lý luận chi tiết HOÀN TOÀN bằng tiếng Việt về tính xác thực, thái độ và cảm xúc. Tuyệt đối không dùng tiếng Anh.

Văn bản: Vắc-xin COVID gây vô sinh ở phụ nữ trẻ và biến đổi gen ở trẻ em."""

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": test_prompt}]
}]

# ✅ FIX: Tách 2 bước để tránh processor truyền text=None nội bộ.
# Bước 1: Chỉ format thành string, KHÔNG tokenize.
formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Bước 2: Tokenize thủ công với keyword text= tường minh.
# Gemma4Processor.__call__ có signature (self, images, text, audio, videos, ...)
# → PHẢI dùng text= để tránh string bị truyền vào tham số images.
_device = next(model.parameters()).device
model_inputs = tokenizer(text=formatted_text, return_tensors="pt").to(_device)

inputs = model_inputs["input_ids"]
attention_mask = model_inputs["attention_mask"]

print("Generating test response...")
model.eval()  # Set eval mode thủ công, KHÔNG dùng for_inference()
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        attention_mask=attention_mask,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        use_cache=True,
    )
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("\n=== TEST OUTPUT ===")
print(response)

In [ ]:
# ✅ HỢP NHẤT VÀ XUẤT ĐỊNH DẠNG GGUF CHO LOCAL INFERENCE
# Sử dụng save_pretrained_gguf của Unsloth để lượng tử hóa và nén mô hình xuống còn ~3-4GB,
# giúp tối ưu hóa bộ nhớ khi chạy offline với LM Studio hoặc Ollama trên máy cá nhân.

gguf_method = "q4_k_m"

# 1. Xuất file GGUF ra ổ đĩa cục bộ trên Kaggle/Colab
print("Đang xuất file GGUF ra ổ đĩa cục bộ...")
model.save_pretrained_gguf("vaccinenlp_gemma4_gguf_local", tokenizer, quantization_method = gguf_method)

# 2. Đẩy thẳng file GGUF lên Hugging Face Hub
print(f"Đang đẩy file GGUF lên Hugging Face Hub tại repo: {MERGED_REPO}...")
model.push_to_hub_gguf(
    MERGED_REPO,
    tokenizer,
    quantization_method = gguf_method,
    token = HF_TOKEN,
)
print("✅ Xuất và đẩy file GGUF thành công!")

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)

create_repo(
    MERGED_REPO,
    token=HF_TOKEN,
    repo_type="model",
    private=False,
    exist_ok=True,
)
print(f"✅ Repo ready: https://huggingface.co/{MERGED_REPO}")

In [ ]:
# ✅ Lưu ý: Việc đẩy file GGUF đã được thực hiện trực tiếp bởi model.push_to_hub_gguf ở trên.
print(f"Mô hình GGUF đã được tải thành công lên Hugging Face Hub tại: https://huggingface.co/{MERGED_REPO}")
print("Bây giờ bạn có thể tải tệp tin .gguf về máy tính cá nhân để chạy trực tiếp trên LM Studio!")

In [ ]:
# ✅ Cleanup: Giải phóng bộ nhớ disk trên Kaggle/Colab sau khi upload
import shutil
import os

print("🧹 Freeing disk space after upload...")

# Xoá thư mục GGUF cục bộ
gguf_local_dir = "vaccinenlp_gemma4_gguf_local-q4_k_m.gguf"
if os.path.exists(gguf_local_dir):
    os.remove(gguf_local_dir)
    print(f"✅ Deleted {gguf_local_dir}")

# Xoá HF hub cache
hf_cache = "/root/.cache/huggingface/hub"
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print(f"✅ Deleted {hf_cache}")

In [ ]:
MODEL_CARD = """---\nlicense: gemma\nbase_model: unsloth/gemma-4-E4B-it\ntags:\n  - vietnamese\n  - vaccine\n  - misinformation\n  - public-health\n  - explainable-ai\n  - qlora-merged\nlanguage:\n  - vi\npipeline_tag: text-generation\ninference: true\n---\n\n# VaccineNLP — Gemma-4 E4B Reasoning Engine (Merged)\n\nMerged version of [hung2903/gemma-4-E4B-unsloth-vaccine-xai](https://huggingface.co/hung2903/gemma-4-E4B-unsloth-vaccine-xai) QLoRA adapter merged into base [unsloth/gemma-4-E4B-it](https://huggingface.co/unsloth/gemma-4-E4B-it).\n\n## Purpose\n\n**XAI Reasoning Engine** for VaccineNLP system — Chain-of-Thought explanations for vaccine misinformation detection in Vietnamese.\n\n## Performance (Gold Test Set, n=186)\n\n| Metric | Score |\n|---|:---:|\n| Macro F1 Misinfo | 0.6377 |\n| Macro F1 Stance | 0.6264 |\n| **Macro F1 Sentiment** | **0.7700** 🥇 |\n| Parse Success Rate | 72.0% |\n\n## Usage\n\n```python\n# Tải file GGUF về máy và nạp vào LM Studio / llama.cpp / Ollama để chạy offline\n# API local sẽ lắng nghe tại: http://localhost:1234/v1\n\nprompt = \'\'\'Bạn là chuyên gia y tế công cộng phân tích nội dung về vắc-xin.\nVăn bản: Vắc-xin COVID gây vô sinh.\nLý luận:\'\'\'\n\n# Gọi qua thư viện openai chuẩn:\n# import openai\n# client = openai.OpenAI(base_url=\"http://localhost:1234/v1\", api_key=\"lm-studio\")\n```\n\n## Citation\n\n```bibtex\n@thesis{vaccinenlp2026,\n  title={Ứng dụng Xử lý Ngôn ngữ Tự nhiên trong phát hiện thông tin sai lệch về vaccine},\n  author={Kim Mạnh Hưng and Đinh Lê Quỳnh Phương},\n  school={Trường Đại học Y tế Công cộng (HUPH)},\n  year={2026},\n}\n```\n"""

from io import BytesIO

api.upload_file(
    path_or_fileobj=BytesIO(MODEL_CARD.encode("utf-8")),
    path_in_repo="README.md",
    repo_id=MERGED_REPO,
    repo_type="model",
)
print("✅ Model card uploaded")

In [ ]:
import requests
import time

print("⏳ Đợi 5-10 phút cho HF index model...")
time.sleep(300)

API_URL = f"https://api-inference.huggingface.co/models/{MERGED_REPO}"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

test_prompt = """<start_of_turn>user
Bạn là chuyên gia y tế công cộng. Phân tích bằng tiếng Việt.

Văn bản: Vắc-xin COVID gây vô sinh.

Lý luận:<end_of_turn>
<start_of_turn>model
"""

response = requests.post(
    API_URL,
    headers=headers,
    json={
        "inputs": test_prompt,
        "parameters": {"max_new_tokens": 200, "temperature": 0.3, "return_full_text": False},
        "options": {"wait_for_model": True},
    },
    timeout=120,
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    print(f"✅ Response: {response.json()}")
    print("\n🎉 HF Inference API hoạt động! Path A1 thành công!")
else:
    print(f"⚠️ Error: {response.text[:500]}")